In [29]:
import pandas as pd

df = pd.read_parquet("dataset/dataset.parquet")

df.head()

,url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date,__index_level_0__,__index_level_1__,system_prompt,user_prompt
0,https://www.irozhlas.cz/ekonomika/citibank-okd...,[],Soud potvrdil platnost pohledávek Citibank za ...,"Pohledávky za více než deset miliard korun, kt...","[Okd, Citibank, Pohledávky, Ostrava, Soud, Poh...",5,"V roce 2019 ostravský soud rozhodl, že pohledá...",NaN,6,Ekonomika,[],0,3,2021-09-22,7,178,Pro zadaný text pocházející ze zpravodajského ...,Vygeneruj klasifikaci pro následující příklad:...
1,https://www.novinky.cz/ekonomika/clanek/lego-c...,[],Lego chce dát vale plastovým obalům. Pilotní p...,"Lego bude balit stavebnice do papíru, nahradí ...",[],5,Na zavádění nového typu balení a s tím souvise...,NaN,4,Ekonomika,[],0,1,2021-02-22,7,133,Pro zadaný text pocházející ze zpravodajského ...,Vygeneruj klasifikaci pro následující příklad:...
2,https://www.idnes.cz/ekonomika/domaci/hypoteka...,[Martin Petříček],"Hypoteční mejdan končí, úvěry podraží a klesne...",Objem poskytnutých hypoték za letošní rok míří...,"[Česká spořitelna, Čnb - česká národní banka, ...",5,Nad touto hranicí byla průměrná úroková sazba ...,85.0,2,Ekonomika,[1],1,4,2021-09-23,7,59,Pro zadaný text pocházející ze zpravodajského ...,Vygeneruj klasifikaci pro následující příklad:...
3,https://www.novinky.cz/ekonomika/clanek/k-milo...,[Pavel Cechl],K Milostivému létu se přidaly další banky,"Možností, jak se v rámci tzv. Milostivého léta...",[],5,Milostivé léto je novelou exekučního řádu urče...,NaN,4,Ekonomika,[1],1,4,2021-11-25,7,116,Pro zadaný text pocházející ze zpravodajského ...,Vygeneruj klasifikaci pro následující příklad:...
4,https://www.idnes.cz/zpravy/zahranicni/vakcina...,[],V evropských zemích se hromadí vakcína AstraZe...,Zemím Evropské unie se ve skladech hromadí nev...,"[Francie, Itálie, Španělsko, Německo, Evropská...",1,Například Francie do pátku zužitkovala 16 proc...,778.0,2,Zahraničí,[],0,6,2021-02-27,1,127,Pro zadaný text pocházející ze zpravodajského ...,Vygeneruj klasifikaci pro následující příklad:...


In [30]:
df = df.drop(columns=[
    "system_prompt", "user_prompt",               
    "__index_level_0__", "__index_level_1__",     
    "comments_num",                               
    "authors_gender", "authors_cum_gender",       
    "day_of_week", "category_unclean"                                
])

In [35]:
CATEGORY = {1: "Zahraniční", 2: "Domácí", 3: "Sport", 4: "Kultura", 5: "Ekonomika"}
df["category_name"] = df["category"].map(CATEGORY)
df["category"].value_counts()

category
5    200
1    200
4    200
2    200
3    200
Name: count, dtype: int64

In [38]:
SERVER = {1: "seznamzpravy", 2: "idnes", 3: "aktualne", 4: "novinky", 5: "denik", 6: "irozhlas"}
df["source"] = df["server"].map(SERVER)
df["source"].value_counts()

source
idnes           278
denik           209
aktualne        175
seznamzpravy    166
irozhlas        127
novinky          45
Name: count, dtype: int64

In [40]:
df["doc_id"] = range(len(df))

In [43]:
df = df.drop(columns=["server", "category"])

In [44]:
df = df.rename(columns={"category_name": "category"})

In [47]:
df.head()

,url,authors,headline,brief,keywords,content,date,category,source,doc_id
0,https://www.irozhlas.cz/ekonomika/citibank-okd...,[],Soud potvrdil platnost pohledávek Citibank za ...,"Pohledávky za více než deset miliard korun, kt...","[Okd, Citibank, Pohledávky, Ostrava, Soud, Poh...","V roce 2019 ostravský soud rozhodl, že pohledá...",2021-09-22,Ekonomika,irozhlas,0
1,https://www.novinky.cz/ekonomika/clanek/lego-c...,[],Lego chce dát vale plastovým obalům. Pilotní p...,"Lego bude balit stavebnice do papíru, nahradí ...",[],Na zavádění nového typu balení a s tím souvise...,2021-02-22,Ekonomika,novinky,1
2,https://www.idnes.cz/ekonomika/domaci/hypoteka...,[Martin Petříček],"Hypoteční mejdan končí, úvěry podraží a klesne...",Objem poskytnutých hypoték za letošní rok míří...,"[Česká spořitelna, Čnb - česká národní banka, ...",Nad touto hranicí byla průměrná úroková sazba ...,2021-09-23,Ekonomika,idnes,2
3,https://www.novinky.cz/ekonomika/clanek/k-milo...,[Pavel Cechl],K Milostivému létu se přidaly další banky,"Možností, jak se v rámci tzv. Milostivého léta...",[],Milostivé léto je novelou exekučního řádu urče...,2021-11-25,Ekonomika,novinky,3
4,https://www.idnes.cz/zpravy/zahranicni/vakcina...,[],V evropských zemích se hromadí vakcína AstraZe...,Zemím Evropské unie se ve skladech hromadí nev...,"[Francie, Itálie, Španělsko, Německo, Evropská...",Například Francie do pátku zužitkovala 16 proc...,2021-02-27,Zahraniční,idnes,4


In [50]:
def is_empty(x):
    if isinstance(x, str):
        return x.strip() == ""          
    try:
        return len(x) == 0              
    except TypeError:
        return pd.isna(x)               

empty = df.apply(lambda col: col.apply(is_empty).sum())
empty

url           0
authors     390
headline      0
brief         0
keywords    172
content       0
date          0
category      0
source        0
doc_id        0
dtype: int64

In [48]:
df.to_parquet("dataset/articles.parquet", index=False)